Load the dataset

In [2]:
import pandas as pd

# Load the synthetic healthcare data
df = pd.read_csv("/content/synthetic_datas.csv")
# Create a working copy
df_clean = df.copy()

# Display column names to verify
print("Columns in dataset:")
print(df_clean.columns.tolist())
print(f"\nDataset loaded: {df_clean.shape[0]} rows \u00d7 {df_clean.shape[1]} columns")

Columns in dataset:
['Patient_ID', 'Age', 'Gender', 'Ethnicity', 'Weight', 'Height_cm', 'Diagnosis_Date', 'Diagnosis_Code', 'Glucose_mg_dL', 'Risk', 'Patient_Name', 'EmailID']

Dataset loaded: 200 rows × 12 columns


Remove PII

In [3]:
# Remove PII columns
pii_cols = ['Patient_ID', 'Name', 'Address', 'Phone', 'Email']
df_clean = df_clean.drop(columns=pii_cols, errors='ignore')

# Show remaining columns
print("Columns after removing PII:")
print(df_clean.columns.tolist())
print("Total columns:", len(df_clean.columns))

Columns after removing PII:
['Age', 'Gender', 'Ethnicity', 'Weight', 'Height_cm', 'Diagnosis_Date', 'Diagnosis_Code', 'Glucose_mg_dL', 'Risk', 'Patient_Name', 'EmailID']
Total columns: 11


List columns to remove

These are sensitive (PII) columns you don’t want in your dataset.

Removes those columns.
If some don’t exist, it won’t crash (thanks to errors='ignore').

Print remaining columns and count to confirm removal.

## Exercise 3: Drop duplicate rows

Check for and remove any duplicate rows in the dataset. Report how many duplicates were found and removed.

In [4]:
# Remove duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

# Show results
print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

Rows before: 200
Rows after: 200
Duplicates removed: 0


Count rows before cleaning
Remove duplicates using .drop_duplicates()
Count rows after cleaning
Compare counts to see how many duplicates were removed

Handle missing values in numeric columns

Select numeric columns automatically
Avoids manually listing columns.
Check missing values
See what needs fixing before cleaning.
Convert data safely(Sometimes columns look numeric but are actually stored as strings (text))
errors='coerce' turns bad values('unknown', 'N/A', '??') into NaN.
Fill missing values (imputation)
Replace all missing values with the median in one step.
Verify results
Confirm no missing values remain.

In [5]:
# Get numeric columns
numeric_cols = df_clean.select_dtypes(include='number').columns

print("Numeric columns:", list(numeric_cols))

# Missing values before
print("\nMissing before:\n", df_clean[numeric_cols].isna().sum())

# Convert to numeric (invalid → NaN) and fill with median
df_clean[numeric_cols] = df_clean[numeric_cols].apply(pd.to_numeric, errors='coerce')
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())

# Missing values after
print("\nMissing after:\n", df_clean[numeric_cols].isna().sum())
print("\n✓ Missing values handled!")

Numeric columns: ['Age', 'Height_cm', 'Glucose_mg_dL', 'Risk']

Missing before:
 Age              28
Height_cm        40
Glucose_mg_dL    42
Risk              0
dtype: int64

Missing after:
 Age              0
Height_cm        0
Glucose_mg_dL    0
Risk             0
dtype: int64

✓ Missing values handled!


Standardize categorical variables

Inspect data first
See all variations (e.g., M, male, Female, etc.).
Clean text
Remove spaces → .str.strip()
Make consistent → .str.lower()
Fix known variations
Map short forms (m, f) to full values.
Standardize format
Capitalize → .str.capitalize()
Verify results
Check value counts again.

In [6]:
# Before
print("Before standardization - Gender:")
print(df_clean['Gender'].value_counts(dropna=False))

Before standardization - Gender:
Gender
M         46
F         40
NaN       38
Female    36
Male      33
Other      7
Name: count, dtype: int64


In [8]:
import numpy as np

# Clean and standardize
df_clean['Gender'] = (
    df_clean['Gender']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': np.nan, 'none': np.nan, 'm': 'male', 'f': 'female'})
    .str.capitalize()
)

# After
print("\nAfter standardization - Gender:")
print(df_clean['Gender'].value_counts(dropna=False))


After standardization - Gender:
Gender
Male      79
Female    76
NaN       38
Other      7
Name: count, dtype: int64
